# Kairos: данные в Colab

1. Копирует тома core-архива с Drive авторов в **твой** Drive на стороне сервера (параллельно, без скачивания через твой интернет) и сверяет SHA-256.
2. Распаковывает лёгкую часть (тексты, метки, manifest) и кладёт её в `kairos_work/light.tar`, чтобы следующие ноутбуки не читали архив заново.
3. Запускает проверку меток и бейзлайн, печатает структуру файлов (ключи, без текстов постов).

Рантайм: CPU, GPU не нужен.

Пароль от архива берётся из секрета Colab `KAIROS_PW` (значок ключа слева, «Add new secret»,
включить «Notebook access»). Если секрета нет, то из переменной окружения `KAIROS_PW`, и только потом спрашивается.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, shutil, subprocess, getpass, warnings
import numpy as np, pandas as pd

ARCHIVES = '/content/drive/MyDrive/kairos_archives'
ARCH = f'{ARCHIVES}/lab1_core.7z.001'
WORK = '/content/drive/MyDrive/kairos_work'      # результаты, на Drive
DATA = '/content/kairos_data'                     # тексты и метки, локальный диск VM
MEDIA = '/content/kairos_media'                   # временная распаковка медиа
REPO = '/content/kairos-dataset'
os.makedirs(f'{WORK}/img', exist_ok=True)

_pw = None
def password():
    # Секрет Colab, потом переменная окружения, потом ручной ввод; спрашиваем один раз.
    global _pw
    if _pw is None:
        try:
            from google.colab import userdata
            _pw = userdata.get('KAIROS_PW')
        except Exception:
            _pw = os.environ.get('KAIROS_PW') or getpass.getpass('Пароль от архива: ')
    return _pw

def _z(args):
    return ['7z', *args, f'-p{password()}', '-y', '-mmt=on']

def z(*args):
    # 7z синхронно, возвращает stdout.
    r = subprocess.run(_z(args), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-2000:] or r.stdout[-2000:])
    return r.stdout

def z_start(*args):
    # 7z в фоне: распаковка идёт, пока ноутбук занят другим.
    return subprocess.Popen(_z(args), stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)

def z_wait(p):
    _, err = p.communicate()
    if p.returncode != 0:
        raise RuntimeError(err[-2000:])

subprocess.run('which 7z >/dev/null || apt-get -qq install -y p7zip-full', shell=True, check=True)

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from concurrent.futures import ThreadPoolExecutor
import threading

# name: (file id на Drive авторов, sha256 со страницы датасета). Видео (114 GB) не нужно.
CORE = {
    'lab1_core.7z.001': ('1UEuzJOPCCnMlXW0atu9zbsAYdMNWhmXM', '06085b5df908e9f15d7b611a74c1d07674e3433d3f0d4eb319ebfc781ef6de05'),
    'lab1_core.7z.002': ('1YuGKAOXY1SUowWqBanxKnBHG7TB0UOo3', '62361a5b1987099896c2a1519deb547789c66ef6fe1c42fb12dcf7f6df2454ea'),
    'lab1_core.7z.003': ('1sL-tE0Vn4aK76pDcbt0DhpTjCwzaYwyt', '8f827268da54dfac0967e67d0d44be6b34005a8d0f5314a8906dcf5681fed153'),
    'lab1_core.7z.004': ('1BMSLb--9ATnX8r5q2u6HlqB3Dtdas7SA', '8f2b214dab49a32390086b131278edb9b7480aa3937589cacffd722af63234b6'),
    'lab1_core.7z.005': ('1gG-7MP81j-LOUB7CdxU3Qci5R9cbMrW1', '0f0f3762f5f6ede3c33b5d519115da7aea175e9e901efd139988db463e005200'),
}
FIELDS = 'id,name,size,sha256Checksum'

# Клиент googleapiclient не потокобезопасен: у каждого потока свой.
_local = threading.local()
def service():
    if not hasattr(_local, 'svc'):
        _local.svc = build('drive', 'v3', cache_discovery=False)
    return _local.svc

def ensure_folder(name, parent='root'):
    q = (f"name='{name}' and '{parent}' in parents and trashed=false "
         "and mimeType='application/vnd.google-apps.folder'")
    found = service().files().list(q=q, fields='files(id)').execute()['files']
    if found:
        return found[0]['id']
    body = {'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent]}
    return service().files().create(body=body, fields='id').execute()['id']

folder = ensure_folder(os.path.basename(ARCHIVES))
have = {f['name']: f for f in service().files().list(
    q=f"'{folder}' in parents and trashed=false", fields=f'files({FIELDS})').execute()['files']}

def copy_one(item):
    name, (src_id, sha) = item
    f = have.get(name) or service().files().copy(
        fileId=src_id, body={'name': name, 'parents': [folder]},
        fields=FIELDS, supportsAllDrives=True).execute()
    got = f.get('sha256Checksum')
    status = 'OK' if got == sha else ('нет хэша от Drive, перезапусти позже' if got is None else 'НЕ СОВПАДАЕТ')
    return f"{name}  {int(f['size']) / 1e9:5.1f} GB  sha256 {status}"

with ThreadPoolExecutor(len(CORE)) as ex:
    for line in ex.map(copy_one, CORE.items()):
        print(line)
drive.mount('/content/drive', force_remount=True)   # чтобы маунт увидел новые файлы

Если копирование падает с `downloadQuotaExceeded` или `cannotCopyFile`: открой ссылку тома в браузере, «Создать копию» или «Добавить ярлык на Диск» в папку `kairos_archives`, и перезапусти ячейку.

In [ ]:
# Оглавление архива и лёгкая часть релиза (тексты, метки, manifest).
# Первый раз из архива, дальше из кеша на Drive: распаковка tar занимает секунды.
TOC, LIGHT = f'{WORK}/toc.parquet', f'{WORK}/light.tar'
if os.path.exists(TOC):
    toc = pd.read_parquet(TOC)
else:
    rows, cur = [], {}
    for line in z('l', '-ba', '-slt', ARCH).splitlines():
        if line.startswith('Path = '):
            cur = {'path': line[7:]}
            rows.append(cur)
        elif line.startswith('Size = ') and cur:
            cur['size'] = int(line[7:] or 0)
    toc = pd.DataFrame(rows)
    toc.to_parquet(TOC)
PREFIX = toc.loc[toc['path'].str.endswith('labels.csv'), 'path'].iloc[0][:-len('labels.csv')]

if not os.path.isdir(DATA):
    if os.path.exists(LIGHT):
        subprocess.run(['tar', 'xf', LIGHT, '-C', '/content'], check=True)
    else:
        wanted = [PREFIX + p for p in ('labels.csv', 'split.json', 'media/manifest.jsonl', 'posts', 'authors')]
        z('x', ARCH, f'-o{DATA}', *wanted)
        subprocess.run(['tar', 'cf', LIGHT, '-C', '/content', 'kairos_data'], check=True)
DATA_DIR = os.path.join(DATA, PREFIX) if PREFIX else DATA

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Danil482/kairos-dataset.git', REPO], check=True)
sys.path.insert(0, REPO)
from kairos.data import load_posts, load_labels, load_split, load_followers

posts = load_posts(DATA_DIR)          # только свои посты, вместе с неразмеченной историей
print(f'{len(posts):,} own posts, данные в {DATA_DIR}')

In [ ]:
rel = toc['path'].str[len(PREFIX):]
top = rel.str.split('/').str[0]
ext = rel.str.extract(r'\.([A-Za-z0-9]+)$')[0].str.lower()
gb = lambda d: d.assign(GB=d['sum'] / 1e9)
print(gb(toc.assign(top=top).groupby('top')['size'].agg(['count', 'sum'])))
print(gb(toc[top == 'media'].assign(ext=ext).groupby('ext')['size'].agg(['count', 'sum'])))

In [ ]:
%cd {REPO}
!python -m kairos.label --data "$DATA_DIR"
!python baseline.py --data "$DATA_DIR"

In [ ]:
# Структура файлов без содержимого текстов: ключи, типы, длины строк.
def shape(x):
    if isinstance(x, dict):
        return {k: shape(v) for k, v in x.items()}
    if isinstance(x, list):
        return [shape(x[0]), f'... len={len(x)}'] if x else []
    return f'str[{len(x)}]' if isinstance(x, str) else type(x).__name__

feed = json.load(open(sorted(glob.glob(f'{DATA_DIR}/posts/AUTHOR_*.json'))[0]))
print('POST FEED:', json.dumps(shape(feed), indent=1, ensure_ascii=False)[:6000])
man = pd.read_json(f'{DATA_DIR}/media/manifest.jsonl', lines=True)
print('MANIFEST:', man.shape)
print(man['kind'].value_counts().to_string())